<a href="https://colab.research.google.com/github/Eliekh2/MSBA-Projects/blob/main/Nigerian_E_wallet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Data Ingestion

The goal is to load the Nigerian Banking Mobile Dataset from HuggingFace into Colab.

The file format is parquet, this is useful for analytics.


###Installing HuggingFace

In [ ]:
pip install pandas pyarrow huggingface_hub

In [ ]:
pip install fsspec

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

raw_mobile_money.to_parquet(
    "/content/drive/MyDrive/raw_mobile_money.parquet",
    index=False
)

Mounted at /content/drive


In [ ]:
DATA_PATH = "/content/drive/MyDrive/Customer Churn E-Wallets/"

In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import pandas as pd

raw_mobile_money = pd.read_parquet("hf://datasets/electricsheepafrica/nigerian-banking-mobile-money/nigerian_mobile_money_full.parquet")

###Ingestion Validation

Objective:
Verify that the dataset loaded correctly and assess its structure.

Checks performed:

- shape (rows, columns)
- column names
- preview of data
- data types
- missing values
- duplicates

In [ ]:
print(raw_mobile_money.shape)
print(raw_mobile_money.columns.tolist())
raw_mobile_money.head()

(4000000, 13)
['transaction_id', 'wallet_id', 'timestamp', 'transaction_type', 'amount_ngn', 'fee_ngn', 'balance_after_ngn', 'agent_id', 'channel', 'device_os', 'kyc_tier', 'fraud_flag', 'churn_30d']


,transaction_id,wallet_id,timestamp,transaction_type,amount_ngn,fee_ngn,balance_after_ngn,agent_id,channel,device_os,kyc_tier,fraud_flag,churn_30d
0,90e956c0-7e3e-4207-8bbc-90de4c61d05e,WLT-00001530,2024-06-03 20:35:00,airtime,1000.0,10.0,917.204594,,ussd,android,tier2,False,False
1,3cff1814-4796-4e7e-91ff-2cc1240fe8d8,WLT-00001003,2024-04-01 08:34:00,cashin,1150.0,10.0,644.741704,AGT-00004700,ussd,android,tier2,False,False
2,9fb30da5-5e4e-453f-86cc-3cf842d211ac,WLT-00018460,2024-01-08 19:17:00,cashout,4850.0,10.0,3582.585292,AGT-00001615,app,android,tier1,False,False
3,6127f44f-6d23-4d9f-988f-54dcae598f4d,WLT-00000072,2024-04-14 23:10:00,p2p_send,9800.0,98.0,1546.044463,,ussd,android,tier3,False,False
4,f5ca608d-9ddb-40ca-a4ad-39c8474477cd,WLT-00017907,2024-06-14 07:10:00,billpay,1600.0,10.0,4361.698947,,app,feature_phone,tier2,False,False


In [ ]:
print(raw_mobile_money.info())
print(raw_mobile_money.isnull().sum())
print(raw_mobile_money.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000000 entries, 0 to 3999999
Data columns (total 13 columns):
 #   Column             Dtype         
---  ------             -----         
 0   transaction_id     object        
 1   wallet_id          object        
 2   timestamp          datetime64[ns]
 3   transaction_type   object        
 4   amount_ngn         float64       
 5   fee_ngn            float64       
 6   balance_after_ngn  float64       
 7   agent_id           object        
 8   channel            object        
 9   device_os          object        
 10  kyc_tier           object        
 11  fraud_flag         bool          
 12  churn_30d          bool          
dtypes: bool(2), datetime64[ns](1), float64(3), object(7)
memory usage: 343.3+ MB
None
transaction_id       0
wallet_id            0
timestamp            0
transaction_type     0
amount_ngn           0
fee_ngn              0
balance_after_ngn    0
agent_id             0
channel              0
device_o

###Raw Layer Storage

Objective:
Persist the ingested dataset in its original form to create a raw data layer.

Why:

ensures reproducibility
separates ingestion from transformation
preserves original data

In [ ]:
raw_mobile_money.to_parquet(
    "/content/raw_mobile_money.parquet",
    index=False
)

###Staging Layer Creation

Objective:
Create a cleaned version of the dataset for downstream processing.

In [ ]:
stg_mobile_money = raw_mobile_money.copy()

###Data Cleaning & Transformation
Convert Timestamp

Goal: Ensure proper datetime format for time-based analysis

In [ ]:
stg_mobile_money["timestamp"] = pd.to_datetime(
    stg_mobile_money["timestamp"],
    errors="coerce"
)

###Rename Columns

Goal: Standardize naming for consistency across the pipeline

In [ ]:
stg_mobile_money = stg_mobile_money.rename(columns={
    "timestamp": "txn_timestamp"
})

###Remove Duplicates

Goal: Ensure uniqueness of transactions

In [ ]:
stg_mobile_money = stg_mobile_money.drop_duplicates(
    subset=["transaction_id"]
)

###Handle Missing Values

Goal: Remove critical null records

In [ ]:
stg_mobile_money = stg_mobile_money.dropna(
    subset=["wallet_id", "amount_ngn"]
)

###Fix Data Types

Goal: Ensure numeric fields are properly typed

In [ ]:
stg_mobile_money["amount_ngn"] = pd.to_numeric(
    stg_mobile_money["amount_ngn"], errors="coerce"
)

stg_mobile_money["fee_ngn"] = pd.to_numeric(
    stg_mobile_money["fee_ngn"], errors="coerce"
)

###Staging Layer Storage

Objective:
Save the cleaned dataset for downstream analytics and modeling.

In [ ]:
import os

os.makedirs(DATA_PATH + "staging", exist_ok=True)

In [ ]:
stg_mobile_money.to_parquet(
    DATA_PATH + "staging/stg_mobile_money.parquet",
    index=False
)